# Phase 4: Protein-Protein Interaction Network Analysis

This notebook starts from the Phase 2 DEG outputs and builds subtype-specific STRING / NetworkX protein interaction networks for Basal, Her2, LumA, and LumB.

The workflow is:
- load significant DEG tables from `results/`
- select the strongest genes per subtype for network construction
- query STRING for interaction edges and cache the results locally
- build NetworkX graphs and identify hub genes with degree and betweenness centrality
- detect communities with a built-in modularity-based method
- save summary tables and static network figures for the final report


## Inputs And Outputs

Expected inputs:
- `results/DEGs_Basal_vs_Normal.csv`
- `results/DEGs_Her2_vs_Normal.csv`
- `results/DEGs_LumA_vs_Normal.csv`
- `results/DEGs_LumB_vs_Normal.csv`

Outputs written by this notebook:
- `results/phase4/selected_genes_<Subtype>.csv`
- `results/phase4/string_edges_<Subtype>.csv`
- `results/phase4/hub_genes_<Subtype>.csv`
- `results/phase4/community_summary_<Subtype>.csv`
- `results/phase4/network_summary.csv`
- `figures/phase4/<Subtype>_ppi_network.png`
- `figures/phase4/network_summary.png`

Note: the first STRING query requires internet access. After that, the cached edge tables can be reused without re-querying the API.


In [1]:
from pathlib import Path
import os
import warnings

mpl_cache = Path.cwd() / ".mplconfig"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

FIG_DIR = Path("figures/phase4")
RES_DIR = Path("results/phase4")
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

TUMOR_SUBTYPES = ["Basal", "Her2", "LumA", "LumB"]
SUBTYPE_COLORS = {
    "Basal": "#d62728",
    "Her2": "#9467bd",
    "LumA": "#1f77b4",
    "LumB": "#ff7f0e",
}

STRING_API_URL = "https://string-db.org/api/json/network"
STRING_SPECIES = 9606
STRING_REQUIRED_SCORE = 700
TOP_GENES_PER_SUBTYPE = 100
FORCE_REFRESH_STRING = False

required_deg_files = {
    subtype: Path(f"results/DEGs_{subtype}_vs_Normal.csv")
    for subtype in TUMOR_SUBTYPES
}
missing = [str(path) for path in required_deg_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing Phase 2 DEG outputs: " + ", ".join(missing) +
        ". Run phase2_differential_expression.ipynb first."
    )


## Load Differential Expression Results

For the network input, we start from the significant subtype-vs-normal DEG tables generated in Phase 2.


In [2]:
deg_results = {
    subtype: pd.read_csv(path)
    for subtype, path in required_deg_files.items()
}

deg_counts = []
for subtype, df in deg_results.items():
    deg_counts.append({
        "Subtype": subtype,
        "Significant DEGs": int(df["significant"].fillna(False).sum()),
        "Median |log2FC|": float(df.loc[df["significant"].fillna(False), "log2FoldChange"].abs().median()),
    })

deg_counts_df = pd.DataFrame(deg_counts).set_index("Subtype")
deg_counts_df


,Significant DEGs,Median |log2FC|
Subtype,,
Basal,5790,2.232928
Her2,5714,2.197068
LumA,4064,2.081997
LumB,5982,2.229334


## Select Candidate Genes For The PPI Networks

To keep the networks readable, we select the strongest significant DEGs per subtype by adjusted p-value and absolute log2 fold change. You can raise or lower `TOP_GENES_PER_SUBTYPE` in the config cell.


In [3]:
def select_network_genes(df: pd.DataFrame, top_n: int = TOP_GENES_PER_SUBTYPE) -> pd.DataFrame:
    ranked = df.copy()
    ranked = ranked.dropna(subset=["gene_name", "padj", "log2FoldChange"])
    ranked = ranked[ranked["significant"].fillna(False)].copy()
    ranked["abs_log2FoldChange"] = ranked["log2FoldChange"].abs()
    ranked = ranked.sort_values(["padj", "abs_log2FoldChange"], ascending=[True, False])

    half_n = top_n // 2
    up = ranked[ranked["log2FoldChange"] > 0].head(half_n)
    down = ranked[ranked["log2FoldChange"] < 0].head(half_n)
    selected = pd.concat([up, down], ignore_index=True)

    if len(selected) < top_n:
        already_selected = set(selected["gene_name"])
        remainder = ranked[~ranked["gene_name"].isin(already_selected)].head(top_n - len(selected))
        selected = pd.concat([selected, remainder], ignore_index=True)

    selected = selected.sort_values(["padj", "abs_log2FoldChange"], ascending=[True, False]).head(top_n).copy()
    return selected[["gene_name", "log2FoldChange", "abs_log2FoldChange", "padj", "baseMean"]]


selected_gene_tables = {}
selection_summary = []

for subtype, df in deg_results.items():
    selected = select_network_genes(df)
    selected_gene_tables[subtype] = selected
    selected.to_csv(RES_DIR / f"selected_genes_{subtype}.csv", index=False)
    selection_summary.append({
        "Subtype": subtype,
        "Selected genes": len(selected),
        "Upregulated": int((selected["log2FoldChange"] > 0).sum()),
        "Downregulated": int((selected["log2FoldChange"] < 0).sum()),
        "Top gene": selected.iloc[0]["gene_name"] if not selected.empty else None,
    })

selection_summary_df = pd.DataFrame(selection_summary).set_index("Subtype")
selection_summary_df


,Selected genes,Upregulated,Downregulated,Top gene
Subtype,,,,
Basal,100,50,50,CCNE1
Her2,100,50,50,NTRK2
LumA,100,50,50,MRAS
LumB,100,50,50,SFRP1


## STRING Query Helpers

This section fetches interaction edges from STRING and caches each subtype's edge list under `results/phase4/`. If the cache exists and `FORCE_REFRESH_STRING = False`, the notebook will reuse the saved file.


In [4]:
def fetch_string_network(genes, species=STRING_SPECIES, required_score=STRING_REQUIRED_SCORE) -> pd.DataFrame:
    if len(genes) < 2:
        return pd.DataFrame(columns=["gene_a", "gene_b", "score"])

    response = requests.post(
        STRING_API_URL,
        data={
            "identifiers": "\r".join(genes),
            "species": species,
            "required_score": required_score,
            "network_type": "functional",
            "caller_identity": "cmps297af_project",
        },
        timeout=60,
    )
    response.raise_for_status()
    edges = pd.DataFrame(response.json())
    if edges.empty:
        return pd.DataFrame(columns=["gene_a", "gene_b", "score"])

    keep_cols = [
        col for col in [
            "preferredName_A", "preferredName_B", "score",
            "nscore", "fscore", "pscore", "ascore", "escore", "dscore", "tscore"
        ] if col in edges.columns
    ]
    edges = edges[keep_cols].copy()
    edges = edges.rename(columns={"preferredName_A": "gene_a", "preferredName_B": "gene_b"})
    edges = edges.dropna(subset=["gene_a", "gene_b", "score"]).copy()
    edges["score"] = pd.to_numeric(edges["score"], errors="coerce")
    edges = edges.dropna(subset=["score"])
    edges = edges[edges["gene_a"] != edges["gene_b"]].copy()
    edges["edge_key"] = edges.apply(lambda row: "__".join(sorted([row["gene_a"], row["gene_b"]])), axis=1)
    edges = edges.sort_values("score", ascending=False).drop_duplicates("edge_key").drop(columns="edge_key")
    return edges.reset_index(drop=True)


network_edges = {}
edge_summary_rows = []

for subtype, selected in selected_gene_tables.items():
    cache_path = RES_DIR / f"string_edges_{subtype}.csv"
    genes = selected["gene_name"].tolist()

    if cache_path.exists() and not FORCE_REFRESH_STRING:
        edges = pd.read_csv(cache_path)
        source = "cache"
    else:
        try:
            edges = fetch_string_network(genes)
            edges.to_csv(cache_path, index=False)
            source = "live_query"
        except requests.RequestException as exc:
            if cache_path.exists():
                edges = pd.read_csv(cache_path)
                source = f"cache_after_error: {exc}"
            else:
                edges = pd.DataFrame(columns=["gene_a", "gene_b", "score"])
                source = f"query_failed: {exc}"

    network_edges[subtype] = edges
    edge_summary_rows.append({
        "Subtype": subtype,
        "STRING edges": len(edges),
        "Source": source,
        "Mean score": float(edges["score"].mean()) if "score" in edges and not edges.empty else np.nan,
    })

edge_summary_df = pd.DataFrame(edge_summary_rows).set_index("Subtype")
edge_summary_df


,STRING edges,Source,Mean score
Subtype,,,
Basal,200,cache,0.856925
Her2,153,cache,0.851889
LumA,10,cache,0.901700
LumB,307,cache,0.853684


## Build Graphs, Score Hub Genes, And Detect Communities

We use weighted degree and betweenness centrality to identify hub genes. For community detection, the notebook uses `greedy_modularity_communities`, which is built into NetworkX and works without extra dependencies.


In [5]:
def build_graph(selected_table: pd.DataFrame, edges: pd.DataFrame) -> nx.Graph:
    graph = nx.Graph()
    for row in selected_table.itertuples(index=False):
        graph.add_node(
            row.gene_name,
            log2FoldChange=float(row.log2FoldChange),
            abs_log2FoldChange=float(row.abs_log2FoldChange),
            padj=float(row.padj),
            baseMean=float(row.baseMean),
        )

    for row in edges.itertuples(index=False):
        if row.gene_a in graph and row.gene_b in graph:
            score = float(row.score)
            graph.add_edge(
                row.gene_a,
                row.gene_b,
                weight=score,
                distance=1.0 / max(score, 1e-6),
            )
    return graph


def detect_communities(graph: nx.Graph) -> dict[str, int]:
    if graph.number_of_nodes() == 0:
        return {}
    if graph.number_of_edges() == 0:
        return {node: idx + 1 for idx, node in enumerate(graph.nodes())}

    communities = list(nx.algorithms.community.greedy_modularity_communities(graph, weight="weight"))
    mapping = {}
    for idx, community in enumerate(sorted(communities, key=len, reverse=True), start=1):
        for node in community:
            mapping[node] = idx
    return mapping


graphs = {}
hub_tables = {}
network_summary_rows = []

for subtype in TUMOR_SUBTYPES:
    selected = selected_gene_tables[subtype]
    edges = network_edges[subtype]
    graph = build_graph(selected, edges)
    graphs[subtype] = graph

    if graph.number_of_nodes() > 1:
        degree_centrality = nx.degree_centrality(graph)
    else:
        degree_centrality = {node: 0.0 for node in graph.nodes()}

    if graph.number_of_edges() > 0:
        betweenness = nx.betweenness_centrality(graph, weight="distance")
        closeness = nx.closeness_centrality(graph, distance="distance")
        communities = detect_communities(graph)
    else:
        betweenness = {node: 0.0 for node in graph.nodes()}
        closeness = {node: 0.0 for node in graph.nodes()}
        communities = detect_communities(graph)

    hub_df = pd.DataFrame({
        "gene_name": list(graph.nodes()),
        "weighted_degree": [graph.degree(node, weight="weight") for node in graph.nodes()],
        "degree": [graph.degree(node) for node in graph.nodes()],
        "degree_centrality": [degree_centrality[node] for node in graph.nodes()],
        "betweenness_centrality": [betweenness[node] for node in graph.nodes()],
        "closeness_centrality": [closeness[node] for node in graph.nodes()],
        "community_id": [communities.get(node, 0) for node in graph.nodes()],
        "log2FoldChange": [graph.nodes[node]["log2FoldChange"] for node in graph.nodes()],
        "abs_log2FoldChange": [graph.nodes[node]["abs_log2FoldChange"] for node in graph.nodes()],
        "padj": [graph.nodes[node]["padj"] for node in graph.nodes()],
    })

    hub_df = hub_df.sort_values(
        ["weighted_degree", "betweenness_centrality", "abs_log2FoldChange"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    hub_tables[subtype] = hub_df
    hub_df.to_csv(RES_DIR / f"hub_genes_{subtype}.csv", index=False)

    community_summary = (
        hub_df.groupby("community_id", dropna=False)
        .agg(
            module_size=("gene_name", "size"),
            mean_weighted_degree=("weighted_degree", "mean"),
            mean_abs_log2FoldChange=("abs_log2FoldChange", "mean"),
        )
        .sort_values(["module_size", "mean_weighted_degree"], ascending=[False, False])
        .reset_index()
    )
    community_summary.to_csv(RES_DIR / f"community_summary_{subtype}.csv", index=False)

    connected_nodes = sum(1 for _, degree in graph.degree() if degree > 0)
    largest_component = max((len(component) for component in nx.connected_components(graph)), default=0)
    network_summary_rows.append({
        "Subtype": subtype,
        "Selected genes": graph.number_of_nodes(),
        "Connected genes": connected_nodes,
        "Edges": graph.number_of_edges(),
        "Density": nx.density(graph) if graph.number_of_nodes() > 1 else 0.0,
        "Communities": int(hub_df["community_id"].nunique()) if not hub_df.empty else 0,
        "Largest component": largest_component,
        "Top hub gene": hub_df.iloc[0]["gene_name"] if not hub_df.empty else None,
    })

network_summary_df = pd.DataFrame(network_summary_rows).set_index("Subtype")
network_summary_df.to_csv(RES_DIR / "network_summary.csv")
network_summary_df


,Selected genes,Connected genes,Edges,Density,Communities,Largest component,Top hub gene
Subtype,,,,,,,
Basal,100,39,187,0.037778,66,37,CCNB1
Her2,100,41,153,0.030909,65,30,CDK1
LumA,100,14,10,0.002020,92,5,ESR1
LumB,100,49,307,0.062020,57,43,CDK1


## Visualize The Networks

The node size is scaled by weighted degree, the node color reflects community membership, and the labels highlight the top hub genes in each subtype network.


In [6]:
def plot_subtype_network(subtype: str, graph: nx.Graph, hub_df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_axis_off()

    if graph.number_of_nodes() == 0:
        ax.text(0.5, 0.5, f"{subtype}: no selected genes available", ha="center", va="center")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"{subtype}_ppi_network.png", dpi=180, bbox_inches="tight")
        plt.show()
        return

    if graph.number_of_edges() == 0:
        pos = nx.circular_layout(graph)
    else:
        pos = nx.spring_layout(graph, seed=42, weight="weight", k=1.3 / np.sqrt(max(graph.number_of_nodes(), 1)))

    hub_lookup = hub_df.set_index("gene_name")
    node_sizes = hub_lookup.loc[list(graph.nodes()), "weighted_degree"].fillna(0).to_numpy()
    node_sizes = 200 + 900 * (node_sizes / node_sizes.max()) if node_sizes.max() > 0 else np.full_like(node_sizes, 250)
    community_ids = hub_lookup.loc[list(graph.nodes()), "community_id"].fillna(0).astype(int).to_numpy()
    cmap = plt.cm.get_cmap("tab20", max(community_ids.max(), 1) + 1)
    node_colors = [cmap(cid) for cid in community_ids]
    edge_widths = [1.0 + 3.0 * graph[u][v].get("weight", 0.0) for u, v in graph.edges()]

    nx.draw_networkx_edges(graph, pos, ax=ax, alpha=0.25, width=edge_widths, edge_color="#666666")
    nx.draw_networkx_nodes(
        graph,
        pos,
        ax=ax,
        node_size=node_sizes,
        node_color=node_colors,
        edgecolors="black",
        linewidths=0.4,
    )

    top_labels = hub_df.head(12)["gene_name"].tolist()
    nx.draw_networkx_labels(
        graph,
        pos,
        labels={node: node for node in top_labels},
        font_size=8,
        font_weight="bold",
        ax=ax,
    )

    top_hub = hub_df.iloc[0]["gene_name"] if not hub_df.empty else "None"
    ax.set_title(
        f"{subtype} PPI Network\n"
        f"nodes={graph.number_of_nodes()} | edges={graph.number_of_edges()} | top hub={top_hub}",
        fontsize=13,
    )
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{subtype}_ppi_network.png", dpi=180, bbox_inches="tight")
    plt.show()


for subtype in TUMOR_SUBTYPES:
    plot_subtype_network(subtype, graphs[subtype], hub_tables[subtype])


summary_plot_df = network_summary_df.reset_index()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=summary_plot_df, x="Subtype", y="Edges", color="#4c78a8", ax=axes[0])
sns.barplot(data=summary_plot_df, x="Subtype", y="Connected genes", color="#f58518", ax=axes[1])
sns.barplot(data=summary_plot_df, x="Subtype", y="Communities", color="#54a24b", ax=axes[2])
axes[0].set_title("PPI edges")
axes[1].set_title("Connected genes")
axes[2].set_title("Detected communities")
for ax in axes:
    ax.set_xlabel("")
plt.suptitle("Phase 4 Network Summary Across Breast Cancer Subtypes", y=1.05, fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / "network_summary.png", dpi=180, bbox_inches="tight")
plt.show()


## Interpretation Prompts

When you review the output tables and figures, focus on:
- which subtype has the densest or most fragmented PPI network
- which hub genes appear in the highest-centrality positions within each subtype
- whether the detected communities align with the GO terms from Phase 3
- whether the same hub genes are already known subtype markers in the breast cancer literature

A good next reporting step is to take the top 5 to 10 hub genes per subtype from `results/phase4/hub_genes_<Subtype>.csv` and write a short biological interpretation for each network.
